Detecteur de Déchets 

In [2]:
import tensorflow as tf
import os

IMG_SIZE = 640 # Défini par le prétraitement Roboflow
NUM_CLASSES = 11 # Défini dans data.yaml

def parse_single_yolo_label(label_path):
    """
    Lit un fichier texte YOLO contenant un seul objet. type [class_id, x_center, y_center, width, height]
    Retourne la boîte englobante (y_min, x_min, y_max, x_max) et l'ID de la classe.
    """
    # Lecture du fichier
    text = tf.io.read_file(label_path)
    lines = tf.strings.split(text, sep='\n')
    
    # Si le fichier est vide (faille dans le dataset), on retourne des valeurs par défaut
    def process_line():
        parts = tf.strings.split(lines[0])
        cls_id = tf.strings.to_number(parts[0], out_type=tf.int32)
        x_center = tf.strings.to_number(parts[1], out_type=tf.float32)
        y_center = tf.strings.to_number(parts[2], out_type=tf.float32)
        width = tf.strings.to_number(parts[3], out_type=tf.float32)
        height = tf.strings.to_number(parts[4], out_type=tf.float32)

        x_min = x_center - (width / 2.0)
        y_min = y_center - (height / 2.0)
        x_max = x_center + (width / 2.0)
        y_max = y_center + (height / 2.0)
        
        return tf.stack([y_min, x_min, y_max, x_max]), cls_id

    def empty_default():
        return tf.zeros([4], dtype=tf.float32), tf.constant(0, dtype=tf.int32)

    return tf.cond(tf.size(lines) > 0, process_line, empty_default)

def load_image_and_labels(image_path):
    # 1. Image
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3) 
    img = tf.cast(img, tf.float32) / 255.0 
    img = tf.reshape(img, [IMG_SIZE, IMG_SIZE, 3])
    
    # 2. Label
    # Remplacement conditionnel pour gérer les chemins de train, valid et test
    label_path = tf.strings.regex_replace(image_path, "images", "labels")
    label_path = tf.strings.regex_replace(label_path, r"\.jpg$", ".txt")
    
    bbox, cls_id = parse_single_yolo_label(label_path)
    
    # 3. Format de sortie ciblé
    # On renvoie un dictionnaire pour les étiquettes car notre futur réseau aura deux sorties
    return img, {"bbox_output": bbox, "class_output": cls_id}

I0000 00:00:1791962969.108819   47271 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791962969.674709   47271 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791962971.809600   47271 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def build_dataset(split_name):
    file_pattern = f"{split_name}/images/*.jpg" 
    
    dataset = tf.data.Dataset.list_files(file_pattern, shuffle=(split_name == "train"))
    dataset = dataset.map(load_image_and_labels, num_parallel_calls=AUTOTUNE)
    
    if split_name == "train":
        # Cache en mémoire après le premier epoch pour accélérer les suivants
        dataset = dataset.cache()
        dataset = dataset.shuffle(1000)
        
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset

train_ds = build_dataset("data/train")
val_ds   = build_dataset("data/valid")
test_ds  = build_dataset("data/test")

# Vérification
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)
    print("BBox shape:", labels["bbox_output"].shape)
    print("Class shape:", labels["class_output"].shape)

E0000 00:00:1791962975.646616   47271 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



Images shape: (32, 640, 640, 3)
BBox shape: (32, 4)
Class shape: (32,)


Lyse 

In [2]:
import glob
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import yaml

IMG_SIZE = 640
MAX_BOXES = 5
BATCH_SIZE = 16

DATA_YAML_PATH = 'data/data.yaml'
with open(DATA_YAML_PATH, 'r', encoding='utf-8') as yaml_file:
  dataset_config = yaml.safe_load(yaml_file)

CLASS_NAMES = dataset_config['names']
CLASS_NAMES


['cheops',
 'debris',
 'double_start',
 'earth_observation_sat_1',
 'lisa_pathfinder',
 'proba_2',
 'proba_3_csc',
 'proba_3_ocs',
 'smart_1',
 'soho',
 'xmm_newton']

Préparation d'un dataframe de reconnaissance 

In [5]:
def create_dataset_dataframe(image_dir):
  image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
  records = []

  for img_path in image_paths:
    label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')
    boxes = []
    classes_present = []

    if os.path.exists(label_path):
      with open(label_path, 'r') as f:
        for line in f.readlines():
          parts = line.strip().split()
          if len(parts) >= 5:
            cls_id = int(parts[0])
            x_center, y_center, w, h = map(float, parts[1:5])

            # Conversion YOLO -> [y_min, x_min, y_max, x_max, cls_id]
            x_min = x_center - (w / 2.0)
            y_min = y_center - (h / 2.0)
            x_max = x_center + (w / 2.0)
            y_max = y_center + (h / 2.0)

            boxes.append([y_min, x_min, y_max, x_max, float(cls_id)])
            if 0 <= cls_id < len(CLASS_NAMES):
              classes_present.append(CLASS_NAMES[cls_id])

    records.append({
        'image_path': img_path,
        'label_path': label_path,
        'has_label': os.path.exists(label_path),
        'num_objects': len(boxes),
        'classes_present': classes_present,
        'boxes_with_class': boxes,
    })

  return pd.DataFrame(records)



In [8]:
# On crée nos DataFrames
df_train = create_dataset_dataframe('data/train/images')
df_val = create_dataset_dataframe('data/valid/images')
df_test = create_dataset_dataframe('data/test/images')
df_test
df_train
df_val



,image_path,label_path,has_label,num_objects,classes_present,boxes_with_class
0,data/valid/images/img030241_jpg.rf.4ffebca7252...,data/valid/labels/img030241_jpg.rf.4ffebca7252...,True,1,[double_start],"[[0.39101562500000003, 0.1765625, 0.823046875,..."
1,data/valid/images/img100649_jpg.rf.d279efa402b...,data/valid/labels/img100649_jpg.rf.d279efa402b...,True,1,[xmm_newton],"[[0.23632812500000003, 0.34335937499999997, 0...."
2,data/valid/images/img041129_jpg.rf.fcf4f8e7250...,data/valid/labels/img041129_jpg.rf.fcf4f8e7250...,True,1,[earth_observation_sat_1],"[[0.56796875, 0.458203125, 0.85703125, 0.86679..."
3,data/valid/images/img000628_jpg.rf.6dc9522a3a0...,data/valid/labels/img000628_jpg.rf.6dc9522a3a0...,True,1,[proba_2],"[[0.19960937499999998, 0.11367187499999998, 0...."
4,data/valid/images/img000558_jpg.rf.ebcbfd72e0c...,data/valid/labels/img000558_jpg.rf.ebcbfd72e0c...,True,1,[proba_2],"[[0.015625, 0.025390625, 0.609375, 0.808984375..."
...,...,...,...,...,...,...
234,data/valid/images/img060533_jpg.rf.bf57257028a...,data/valid/labels/img060533_jpg.rf.bf57257028a...,True,1,[proba_3_csc],"[[0.28046875000000004, 0.26796875, 0.66015625,..."
235,data/valid/images/img040451_jpg.rf.812cfc46f35...,data/valid/labels/img040451_jpg.rf.812cfc46f35...,True,1,[earth_observation_sat_1],"[[0.03828124999999999, 0.1375, 0.33515625, 0.3..."
236,data/valid/images/img022166_jpg.rf.1a934c419d8...,data/valid/labels/img022166_jpg.rf.1a934c419d8...,True,1,[debris],"[[0.521484375, 0.2859375, 0.662890625, 0.47343..."
237,data/valid/images/img050520_jpg.rf.875f221b209...,data/valid/labels/img050520_jpg.rf.875f221b209...,True,1,[lisa_pathfinder],"[[0.15156250000000002, 0.395703125, 0.484375, ..."


TENSORFLOW 

In [7]:
def df_to_tf_dataset(df, is_training=True):
  image_paths = df['image_path'].values
  boxes_list = df['boxes_with_class'].values

  def generator():
    for img_p, box_arr in zip(image_paths, boxes_list):
      # Lecture image via TensorFlow
      img_raw = tf.io.read_file(img_p)
      img = tf.image.decode_jpeg(img_raw, channels=3)
      img = tf.cast(img, tf.float32) / 255.0

      # Padding des boîtes à MAX_BOXES (pour avoir des tailles fixes)
      arr = np.array(box_arr, dtype=np.float32)
      if len(arr) == 0:
        arr = np.zeros((0, 5), dtype=np.float32)

      current_num = len(arr)
      if current_num < MAX_BOXES:
        pad = np.zeros((MAX_BOXES - current_num, 5), dtype=np.float32)
        pad[:, 4] = -1.0  # -1 indique du vide/padding
        arr = np.vstack([arr, pad])
      else:
        arr = arr[:MAX_BOXES]

      yield img, arr

  output_signature = (
      tf.TensorSpec(shape=(640, 640, 3), dtype=tf.float32),
      tf.TensorSpec(shape=(MAX_BOXES, 5), dtype=tf.float32),
  )

  dataset = tf.data.Dataset.from_generator(generator, output_signature=output_signature)

  if is_training:
    dataset = dataset.shuffle(buffer_size=1000)

  dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
  return dataset


# On transforme nos DataFrames en datasets TensorFlow prêts à l'emploi
train_ds = df_to_tf_dataset(df_train, is_training=True)
val_ds = df_to_tf_dataset(df_val, is_training=False)
test_ds = df_to_tf_dataset(df_test, is_training=False)

print("Tout est prêt pour l'entraînement !")

Tout est prêt pour l'entraînement !
